# Laila (`techindro/Laila`) - IndicWhisper LoRA Fine-Tuning on T4 GPU

This notebook mirrors `src/` to fine-tune `ai4bharat/indicwhisper-hi` (fallback: `openai/whisper-small`) using LoRA (`r=32`, `lora_alpha=64`, `q_proj` and `v_proj` only) on a single 16GB T4 GPU.

In [ ]:
import os

os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN")

In [ ]:
!git clone https://github.com/techindro/Laila.git
%cd Laila
!pip install -q -r requirements.txt

In [ ]:
import torch
from src.config import LailaConfig, require_hf_token, resolve_device

hf_token = require_hf_token()
device = resolve_device()
print(f"Active device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from src.train import train_laila

config = LailaConfig(
    max_steps=1000,
    save_steps=250,
    eval_steps=250,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    fp16=True,
    gradient_checkpointing=True,
    hf_token=hf_token,
)

adapter_path = train_laila(
    config=config,
    adapter_name="hinglish",
    max_samples_per_source=1500,
    push_to_hub=True,
    hf_token=hf_token,
)
print(f"Trained adapter saved and pushed to Hub from: {adapter_path}")

In [ ]:
from src.evaluate import evaluate_model_suite, format_benchmark_table

results = evaluate_model_suite(
    config=config,
    adapter_path=str(adapter_path),
    max_samples=100,
    hf_token=hf_token,
)
print(format_benchmark_table(results))

In [ ]:
from src.inference import LailaASR

asr = LailaASR(config=config, hf_token=hf_token, use_vad=True)
print("Available adapters:", asr.adapter_manager.available_adapters())